# Reviewing extracted pipeline results

Loads every `src/pharmas/*/*.parquet` file and checks:

1. Schema — columns, dtypes, union across companies
2. Validity — required fields, `phase` enum, `trial_id`/`source_url`/`extraction_date` formats
3. Sanity — duplicates, empty strings, `company` vs. folder name, phase aliases
4. Coverage — rows/company, phase distribution, missing-field rates
5. Conclusions

Reproducible: re-run top to bottom, no manual steps. Only reads from `src/pharmas/`, writes nothing.

In [1]:
import re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_GLOB = "src/pharmas/*/*.parquet"

paths = sorted(REPO_ROOT.glob(PARQUET_GLOB))
assert paths, f"No parquet files found under {REPO_ROOT / PARQUET_GLOB}"

frames = []
for p in paths:
    df = pd.read_parquet(p)
    df["_source_file"] = str(p.relative_to(REPO_ROOT))
    df["_source_folder"] = p.parent.name
    frames.append(df)

print(f"Found {len(paths)} parquet files:")
for p in paths:
    print(" -", p.relative_to(REPO_ROOT))

Found 20 parquet files:
 - src/pharmas/abbvie/abbvie_pipeline.parquet
 - src/pharmas/amgen/amgen_pipeline.parquet
 - src/pharmas/astrazeneca/astrazeneca_pipeline.parquet
 - src/pharmas/bayer/bayer_pipeline.parquet
 - src/pharmas/bms/bms_pipeline.parquet
 - src/pharmas/boehringer_ingelheim/boehringer_ingelheim_pipeline.parquet
 - src/pharmas/csl/csl_pipeline.parquet
 - src/pharmas/gilead/gilead_pipeline.parquet
 - src/pharmas/gsk/gsk_pipeline.parquet
 - src/pharmas/johnson_johnson/jnj_pipeline.parquet
 - src/pharmas/lilly/lilly_pipeline.parquet
 - src/pharmas/merck_kgaa/merck_kgaa_pipeline.parquet
 - src/pharmas/msd/msd_pipeline.parquet
 - src/pharmas/novartis/novartis_pipeline.parquet
 - src/pharmas/novonordisk/novonordisk_pipeline.parquet
 - src/pharmas/pfizer/pfizer_pipeline.parquet
 - src/pharmas/roche/roche_pipeline.parquet
 - src/pharmas/sanofi/sanofi_pipeline.parquet
 - src/pharmas/takeda/takeda_pipeline.parquet
 - src/pharmas/teva/teva_pipeline.parquet


In [2]:
df

,company,asset_name,synonyms,mechanism_of_action,therapeutic_area,indication,phase,trial_id,source_url,extraction_date,modality,notes,_source_file,_source_folder
0,Teva,denosumab,[Biosimilar to Prolia®],None,None,,Registered,None,https://tevapharm.com/science/pipeline/,2026-07-09,Biosimilar,Launched regional in Europe and approved in US,src/pharmas/teva/teva_pipeline.parquet,teva
1,Teva,golimumab,[Biosimilar to Simponi®],None,None,,Preregistration,None,https://tevapharm.com/science/pipeline/,2026-07-09,Biosimilar,In collaboration with Alvotech for the U.S. ma...,src/pharmas/teva/teva_pipeline.parquet,teva
2,Teva,golimumab,[Biosimilar to Simponi Aria®],None,None,,Preregistration,None,https://tevapharm.com/science/pipeline/,2026-07-09,Biosimilar,In collaboration with Alvotech for the U.S. ma...,src/pharmas/teva/teva_pipeline.parquet,teva
3,Teva,aflibercept,[Biosimilar to Eylea®],None,None,,Preregistration,None,https://tevapharm.com/science/pipeline/,2026-07-09,Biosimilar,In collaboration with Alvotech for the U.S. ma...,src/pharmas/teva/teva_pipeline.parquet,teva
4,Teva,Olanzapine LAI (TEV-'749),None,None,None,Schizophrenia,Preregistration,None,https://tevapharm.com/science/pipeline/,2026-07-09,Small Molecule,Submitted to FDA December 2025,src/pharmas/teva/teva_pipeline.parquet,teva
5,Teva,omalizumab,[Biosimilar to Xolair®],None,None,,Preregistration,None,https://tevapharm.com/science/pipeline/,2026-07-09,Biosimilar,NaN,src/pharmas/teva/teva_pipeline.parquet,teva
6,Teva,denosumab,[Biosimilar to Xgeva®],None,None,,Preregistration,None,https://tevapharm.com/science/pipeline/,2026-07-09,Biosimilar,Launched regional in Europe and under regulato...,src/pharmas/teva/teva_pipeline.parquet,teva
7,Teva,Dual Action Rescue Inhaler (DARI) ICS/SABA (TE...,None,None,None,Asthma,Phase 3,None,https://tevapharm.com/science/pipeline/,2026-07-09,Small Molecule,In collaboration with Launch Therapeutics.,src/pharmas/teva/teva_pipeline.parquet,teva
8,Teva,Duvakitug (TEV-'574),None,None,None,Ulcerative Colitis,Phase 3,None,https://tevapharm.com/science/pipeline/,2026-07-09,Novel Biologic,In collaboration with Sanofi.,src/pharmas/teva/teva_pipeline.parquet,teva
9,Teva,Duvakitug (TEV-'574),None,None,None,Crohn’s Disease,Phase 3,None,https://tevapharm.com/science/pipeline/,2026-07-09,Novel Biologic,In collaboration with Sanofi.,src/pharmas/teva/teva_pipeline.parquet,teva


## 1. Schema

Base schema (`src/schema.py`, `PipelineRecord`) requires: `company`, `asset_name`, `indication`, `phase`.
Optional: `synonyms`, `mechanism_of_action`, `therapeutic_area`, `trial_id`, `source_url`, `extraction_date`, `modality`, `notes`.
`model_config = {"extra": "allow"}` — extra columns (`others`) are permitted per-source extras.

In [3]:
REQUIRED_FIELDS = ["company", "asset_name", "indication", "phase"]
BASE_OPTIONAL_FIELDS = [
    "synonyms", "mechanism_of_action", "therapeutic_area", "trial_id",
    "source_url", "extraction_date", "modality", "notes",
]
KNOWN_EXTRA_FIELDS = ["others"]
META_COLS = {"_source_file", "_source_folder"}

schema_rows = []
for df, p in zip(frames, paths):
    cols = set(df.columns) - META_COLS
    schema_rows.append({
        "folder": p.parent.name,
        "n_rows": len(df),
        "columns": sorted(cols),
        "missing_required": sorted(set(REQUIRED_FIELDS) - cols),
        "unknown_extra": sorted(cols - set(REQUIRED_FIELDS) - set(BASE_OPTIONAL_FIELDS) - set(KNOWN_EXTRA_FIELDS)),
    })

schema_df = pd.DataFrame(schema_rows)
schema_df

,folder,n_rows,columns,missing_required,unknown_extra
0,abbvie,97,"[asset_name, company, extraction_date, indicat...",[],[]
1,amgen,42,"[asset_name, company, extraction_date, indicat...",[],[]
2,astrazeneca,199,"[asset_name, company, extraction_date, indicat...",[],[]
3,bayer,39,"[asset_name, company, extraction_date, indicat...",[],[]
4,bms,91,"[asset_name, company, extraction_date, indicat...",[],[]
5,boehringer_ingelheim,52,"[asset_name, company, extraction_date, indicat...",[],[]
6,csl,34,"[asset_name, company, extraction_date, indicat...",[],[]
7,gilead,50,"[asset_name, company, extraction_date, indicat...",[],[]
8,gsk,76,"[asset_name, company, extraction_date, indicat...",[],[]
9,johnson_johnson,100,"[asset_name, company, extraction_date, indicat...",[],[]


In [4]:
# Combine into one long dataframe (pandas aligns/union columns, filling absent ones with NaN)
data = pd.concat(frames, ignore_index=True, sort=False)
print(f"Combined: {len(data)} rows, {len(paths)} companies, {len(data.columns)} columns")

# dtype per column, per source file — flags a column stored inconsistently across companies
dtype_matrix = pd.DataFrame({
    p.parent.name: pd.read_parquet(p).dtypes.astype(str)
    for p in paths
}).T
dtype_matrix

Combined: 1481 rows, 20 companies, 15 columns


,asset_name,company,extraction_date,indication,mechanism_of_action,modality,notes,others,phase,source_url,synonyms,therapeutic_area,trial_id
abbvie,str,str,str,str,str,str,str,object,str,str,object,str,object
amgen,str,str,object,str,str,str,str,object,str,str,object,str,object
astrazeneca,str,str,str,str,str,str,str,NaN,str,str,object,str,object
bayer,str,str,object,str,str,str,str,object,str,str,object,str,str
bms,str,str,object,str,str,object,object,object,str,str,object,str,object
boehringer_ingelheim,str,str,object,str,str,object,object,NaN,str,str,object,str,object
csl,str,str,object,str,str,object,str,object,str,str,object,str,object
gilead,str,str,str,str,object,object,str,object,str,str,object,str,object
gsk,str,str,str,str,str,NaN,object,object,str,str,object,str,object
johnson_johnson,str,str,str,str,object,object,object,NaN,str,str,object,str,object


## 2. Validity

Required fields non-null/non-empty; `phase` in the controlled enum (`docs/data-model.md`); `trial_id` looks like `NCT########`; `source_url` looks like a URL; `extraction_date` parses and isn't in the future.

In [5]:
def is_blank(series):
    return series.isna() | (series.astype(str).str.strip() == "")

required_issues = []
for field in REQUIRED_FIELDS:
    if field not in data.columns:
        continue
    blank = is_blank(data[field])
    if blank.any():
        bad = data.loc[blank, ["_source_folder", "company", "asset_name", "indication", "phase"]]
        for _, row in bad.iterrows():
            required_issues.append({"issue": f"blank `{field}`", **row.to_dict()})

required_issues_df = pd.DataFrame(required_issues)
print(f"Rows with a blank required field: {len(required_issues_df)}")
required_issues_df

Rows with a blank required field: 11


,issue,_source_folder,company,asset_name,indication,phase
0,blank `indication`,teva,Teva,denosumab,,Registered
1,blank `indication`,teva,Teva,golimumab,,Preregistration
2,blank `indication`,teva,Teva,golimumab,,Preregistration
3,blank `indication`,teva,Teva,aflibercept,,Preregistration
4,blank `indication`,teva,Teva,omalizumab,,Preregistration
5,blank `indication`,teva,Teva,denosumab,,Preregistration
6,blank `indication`,teva,Teva,TEV-'285,,Preclinical
7,blank `indication`,teva,Teva,TEV-'289,,Preclinical
8,blank `indication`,teva,Teva,TEV-'292,,Preclinical
9,blank `indication`,teva,Teva,TEV-'295,,Preclinical


In [6]:
# phase enum (src/schema.py Phase / docs/data-model.md)
VALID_PHASES = {
    "Preclinical", "Phase 1", "Phase 1/2", "Phase 2", "Phase 2/3",
    "Phase 3", "Preregistration", "Registered", "Discontinued",
}

bad_phase = data[~data["phase"].isin(VALID_PHASES)]
print(f"Rows with an out-of-enum `phase`: {len(bad_phase)}")
print("\nPhase value counts (all companies):")
print(data["phase"].value_counts(dropna=False))
bad_phase[["_source_folder", "company", "asset_name", "phase"]]

Rows with an out-of-enum `phase`: 0

Phase value counts (all companies):
phase
Phase 3            518
Phase 2            403
Phase 1            367
Preregistration     99
Registered          73
Discontinued        13
Preclinical          8
Name: count, dtype: int64


,_source_folder,company,asset_name,phase


In [7]:
# trial_id format: NCT + 8 digits
NCT_RE = re.compile(r"^NCT\d{8}$")
has_trial_id = data["trial_id"].notna() & (data["trial_id"].astype(str).str.strip() != "")
bad_trial_id = data[has_trial_id & ~data["trial_id"].astype(str).str.match(NCT_RE)]
print(f"trial_id populated: {has_trial_id.sum()} / {len(data)}")
print(f"trial_id populated but not NCT######## format: {len(bad_trial_id)}")
bad_trial_id[["_source_folder", "company", "asset_name", "trial_id"]]

trial_id populated: 138 / 1481
trial_id populated but not NCT######## format: 0


,_source_folder,company,asset_name,trial_id


In [8]:
# source_url: looks like a URL
has_url = data["source_url"].notna() & (data["source_url"].astype(str).str.strip() != "")
bad_url = data[has_url & ~data["source_url"].astype(str).str.match(r"^https?://")]
print(f"source_url populated: {has_url.sum()} / {len(data)}")
print(f"source_url populated but not http(s):// : {len(bad_url)}")
bad_url[["_source_folder", "company", "asset_name", "source_url"]]

source_url populated: 1481 / 1481
source_url populated but not http(s):// : 0


,_source_folder,company,asset_name,source_url


In [9]:
# extraction_date: parses to a date, not in the future
parsed_date = pd.to_datetime(data["extraction_date"], errors="coerce")
has_date = data["extraction_date"].notna()
unparseable = data[has_date & parsed_date.isna()]
in_future = data[parsed_date.notna() & (parsed_date.dt.date > date.today())]

print(f"extraction_date populated: {has_date.sum()} / {len(data)}")
print(f"unparseable: {len(unparseable)}, in the future (> {date.today()}): {len(in_future)}")
print("\nextraction_date range per company:")
data.assign(extraction_date=parsed_date).groupby("_source_folder")["extraction_date"].agg(["min", "max"])

extraction_date populated: 1481 / 1481
unparseable: 0, in the future (> 2026-07-13): 0

extraction_date range per company:


,min,max
_source_folder,,
abbvie,2026-07-08,2026-07-08
amgen,2026-07-09,2026-07-09
astrazeneca,2026-07-08,2026-07-08
bayer,2026-07-09,2026-07-09
bms,2026-07-09,2026-07-09
boehringer_ingelheim,2026-07-09,2026-07-09
csl,2026-07-09,2026-07-09
gilead,2026-07-09,2026-07-09
gsk,2026-07-08,2026-07-08


## 3. Sanity checks

Exact-duplicate rows, `company` value vs. its folder name, and `indication == asset_name` (known Novo Nordisk anomaly per memory — check if it recurs elsewhere).

In [10]:
# exact duplicate rows (ignoring internal _source_* meta cols)
content_cols = [c for c in data.columns if c not in META_COLS]
# list/array columns aren't hashable for duplicated() — stringify a copy for comparison only
hashable = data[content_cols].apply(lambda col: col.map(lambda v: str(v) if isinstance(v, (list, tuple, np.ndarray)) else v))
dup_mask = hashable.duplicated(keep=False)
print(f"Exact duplicate rows: {dup_mask.sum()}")
data.loc[dup_mask, ["_source_folder", "company", "asset_name", "indication", "phase"]].sort_values("_source_folder")

Exact duplicate rows: 4


,_source_folder,company,asset_name,indication,phase
230,astrazeneca,AstraZeneca,Enhertu,HER2 expressing solid tumours,Phase 3
232,astrazeneca,AstraZeneca,Enhertu,HER2 expressing solid tumours,Phase 3
496,boehringer_ingelheim,Boehringer Ingelheim,Anti-fibrotic agent,Cardiovascular-Renal-Metabolic,Phase 1
504,boehringer_ingelheim,Boehringer Ingelheim,Anti-fibrotic agent,Cardiovascular-Renal-Metabolic,Phase 1


In [11]:
# `company` value should be consistent within a folder (one company per file)
print("Distinct `company` values per source folder:")
print(data.groupby("_source_folder")["company"].unique().apply(list))

Distinct `company` values per source folder:
_source_folder
abbvie                                [AbbVie]
amgen                                  [Amgen]
astrazeneca                      [AstraZeneca]
bayer                                  [Bayer]
bms                     [Bristol Myers Squibb]
boehringer_ingelheim    [Boehringer Ingelheim]
csl                                      [CSL]
gilead                                [Gilead]
gsk                                      [GSK]
johnson_johnson            [Johnson & Johnson]
lilly                              [Eli Lilly]
merck_kgaa                        [Merck KGaA]
msd                              [Merck & Co.]
novartis                            [Novartis]
novonordisk                     [Novo Nordisk]
pfizer                                [Pfizer]
roche                                  [Roche]
sanofi                                [Sanofi]
takeda                                [Takeda]
teva                                    [Teva]


In [12]:
# known anomaly type (Novo Nordisk, per memory): indication text equal to the asset_name itself
same_as_asset = data[data["indication"].astype(str).str.strip() == data["asset_name"].astype(str).str.strip()]
print(f"Rows where indication == asset_name: {len(same_as_asset)}")
same_as_asset[["_source_folder", "company", "asset_name", "indication", "therapeutic_area"]]

Rows where indication == asset_name: 0


,_source_folder,company,asset_name,indication,therapeutic_area


## 4. Coverage

Rows per company, phase distribution per company, and missing-field rate per company (optional fields legitimately vary by source richness — this is descriptive, not a pass/fail).

In [13]:
rows_per_company = data.groupby("_source_folder").size().rename("n_rows").sort_values(ascending=False)
rows_per_company

_source_folder
astrazeneca             199
roche                   128
msd                     109
novartis                106
johnson_johnson         100
abbvie                   97
pfizer                   96
bms                      91
sanofi                   77
lilly                    76
gsk                      76
boehringer_ingelheim     52
gilead                   50
amgen                    42
bayer                    39
takeda                   38
novonordisk              37
csl                      34
teva                     24
merck_kgaa               10
Name: n_rows, dtype: int64

In [14]:
phase_by_company = pd.crosstab(data["_source_folder"], data["phase"])
phase_by_company

phase,Discontinued,Phase 1,Phase 2,Phase 3,Preclinical,Preregistration,Registered
_source_folder,,,,,,,
abbvie,0,25,30,18,0,6,18
amgen,0,6,10,26,0,0,0
astrazeneca,13,40,39,107,0,0,0
bayer,0,19,6,10,0,4,0
bms,0,34,19,35,0,3,0
boehringer_ingelheim,0,24,14,11,0,3,0
csl,0,1,7,6,0,0,20
gilead,0,21,12,12,0,5,0
gsk,0,29,24,17,0,6,0


In [15]:
def is_missing(v):
    if v is None:
        return True
    if isinstance(v, float) and pd.isna(v):
        return True
    if isinstance(v, (list, tuple, np.ndarray)):
        return len(v) == 0
    if isinstance(v, str):
        return v.strip() == ""
    return False

optional_fields = ["mechanism_of_action", "therapeutic_area", "trial_id", "source_url", "extraction_date", "modality", "notes", "synonyms", "others"]
present_optional = [f for f in optional_fields if f in data.columns]

missing_rate = pd.DataFrame({
    field: data.groupby("_source_folder")[field].apply(lambda col: col.map(is_missing).mean())
    for field in present_optional
})
missing_rate.style.format("{:.0%}")

,mechanism_of_action,therapeutic_area,trial_id,source_url,extraction_date,modality,notes,synonyms,others
_source_folder,,,,,,,,,
abbvie,3%,0%,100%,0%,0%,0%,97%,100%,78%
amgen,0%,0%,100%,0%,0%,2%,69%,31%,74%
astrazeneca,7%,7%,100%,0%,0%,7%,93%,57%,100%
bayer,54%,0%,23%,0%,0%,23%,90%,51%,3%
bms,96%,0%,100%,0%,0%,100%,100%,86%,0%
boehringer_ingelheim,0%,0%,100%,0%,0%,100%,100%,63%,100%
csl,26%,0%,100%,0%,0%,100%,94%,35%,0%
gilead,100%,0%,100%,0%,0%,100%,60%,100%,94%
gsk,0%,0%,100%,0%,0%,100%,100%,34%,0%


In [16]:
# therapeutic_area kept verbatim per-source (see memory: normalization deferred) — how fragmented is the vocab so far?
ta_counts = data.groupby("_source_folder")["therapeutic_area"].nunique().rename("n_distinct_therapeutic_area")
print(ta_counts)
print(f"\nTotal distinct therapeutic_area labels across all companies: {data['therapeutic_area'].nunique()}")
data["therapeutic_area"].value_counts().head(20)

_source_folder
abbvie                  6
amgen                   8
astrazeneca             5
bayer                   4
bms                     5
boehringer_ingelheim    6
csl                     5
gilead                  3
gsk                     4
johnson_johnson         3
lilly                   4
merck_kgaa              3
msd                     9
novartis                7
novonordisk             3
pfizer                  4
roche                   6
sanofi                  5
takeda                  5
teva                    0
Name: n_distinct_therapeutic_area, dtype: int64

Total distinct therapeutic_area labels across all companies: 57


therapeutic_area
Oncology                                    477
Immunology                                  170
Neuroscience                                110
Oncology/Hematology                          58
Cardiometabolic Health                       34
Vaccines                                     33
Respiratory & Immunology                     28
Hematology                                   28
Respiratory, Immunology and Inflammation     26
Rare Disease                                 24
Cardiovascular, Renal and Metabolism         23
Infectious Diseases                          22
Cardiovascular, Renal and Metabolic          21
Inflammation & Immunology                    21
Ophthalmology                                19
Oncology: Solid Tumors                       19
Diabetes                                     18
Virology                                     17
Cardiovascular-Renal-Metabolic               14
Obesity                                      14
Name: count, dtype: int

## 5. Conclusions

**Overall: healthy. 20/20 companies loaded, 1481 rows, schema-conformant.** AbbVie, Amgen, AstraZeneca,
Bayer, BMS, Boehringer Ingelheim, CSL, Gilead, GSK, J&J, Eli Lilly, Merck KGaA, Merck & Co./MSD,
Novartis, Novo Nordisk, Pfizer, Roche, Sanofi, Takeda, Teva. Zero out-of-enum `phase` values, zero
malformed `trial_id`/`source_url`, zero unparseable or future `extraction_date`. `company` is
single-valued per file and correctly named per folder.

**Findings, all traced to documented/intentional source handling — not extraction bugs:**

1. **11 blank `indication` rows, all Teva.** Biosimilars (`denosumab`, `golimumab` x2, `aflibercept`,
   `omalizumab`) carry only the "Biosimilars" tag, no indication-specific tag; 5 preclinical codes
   (`TEV-'285/289/292/295/296`) have no tag at all. Documented in `src/pharmas/teva/log.md` point 6 —
   source has no indication text to extract for these rows.
2. **4 exact duplicate rows**, both explained:
   - **AstraZeneca `Enhertu` x2** — traced to source: the raw HTML has two genuinely distinct trials,
     `Enhertu DESTINY-PanTumor03 (China)` and `EnhertuDESTINY-PanTumor02`, both same MoA/indication/phase.
     `astrazeneca_to_parquet.py`'s `_strip_trial_suffix()` deliberately drops the trial-code suffix from
     `asset_name` (with a hardcoded replace for the no-space `EnhertuDESTINY-PanTumor02` case), so both
     collapse to identical rows by design. Intentional normalization choice, not a parsing defect.
   - **Boehringer Ingelheim `Anti-fibrotic agent` x2** — expected per `boehringer_ingelheim/log.md`:
     undisclosed Phase 1 compounds fall back to MoA label as `asset_name` and therapeutic_area as
     `indication`, so two distinct undisclosed compounds sharing an MoA description are indistinguishable
     in BI's own PDF.
3. **`extraction_date` dtype still split**: MSD and Merck KGaA store real `datetime.date` objects;
   the other 18 companies store ISO strings. Same as previously flagged — harmless for
   `pd.to_datetime`, but raw `isinstance(v, date)` checks would only catch those two.
4. **`therapeutic_area` vocabulary more fragmented at full scale**: 57 distinct raw labels across 20
   companies (was 34/9). Confirms the deferred-normalization call was correct — now that all 20 are
   in, this is the point to map to a controlled vocabulary if downstream analysis needs it.
5. **Optional-field coverage remains source-dependent, not a bug**: `trial_id` populated for only
   138/1481 rows (9%), `modality` missing entirely for several companies, `others` absent for a few —
   expected, never backfilled from external sources per data-model.md.

**Checked and clean:** blank `asset_name`/`phase`/`company` (0 across all 1481 rows); the
`indication == asset_name` placeholder anomaly (Novo Nordisk precedent) does not recur anywhere.
**Not covered here:** transposed-column anomalies (Roche/GSK precedent) — needs a semantic spot-check
per company, not a mechanical one.

**Verdict: portfolio is healthy.** No blank `company`/`asset_name`/`phase`, no schema violations. Every
finding above traces to a documented or provably-intentional upstream choice — none is a new regression
from finishing the remaining 11 companies.

**Reproducibility:** this notebook only reads `src/pharmas/*/*.parquet` — safe to rerun top-to-bottom
any time after new companies land; no manual steps, no writes. Re-execute headlessly with:
`uv run jupyter nbconvert --to notebook --execute --inplace notebooks/00_reviewing_results.ipynb`
(requires the `ipykernel`/`nbconvert` dev dependencies added to `pyproject.toml`).